# Reproduce the solar filament predictions

Use the original Kaggle test JPEGs and the locally trained detector and U-Net checkpoints. The replay verifies model, configuration and input identities before regenerating the competition CSV.

In [ ]:
import hashlib
import json
import os
from pathlib import Path
import subprocess
import sys
from tempfile import mkdtemp

root = Path.cwd()
release = json.loads((root / "artifacts" / "release.json").read_text(encoding="utf-8"))
test_images = root / "data" / "MAGFiLO_1.0_Kaggle_2026" / "test" / "test_images"
out_root = root / "output"
out_root.mkdir(parents=True, exist_ok=True)
out = Path(mkdtemp(prefix="notebook-reproduction-", dir=out_root))
env = os.environ.copy()
env.update({"PYTHONPATH": str(root / "src"), "OMP_NUM_THREADS": "1", "MKL_NUM_THREADS": "1"})
def sha256(path):
    with path.open("rb") as f:
        return hashlib.file_digest(f, "sha256").hexdigest()
def run(args):
    subprocess.run([sys.executable, *map(str, args)], cwd=root, env=env, check=True)


## Verify the inputs

The release manifest lists the model, settings and hashes. The test file manifest is an inventory of the organizer-supplied input JPEGs; it does not contain labels or predicted masks.


In [ ]:
for name, digest in release["artifact_sha256"].items():
    assert sha256(root / name) == digest, f"Unexpected file: {name}"
expected = json.loads((root / release["test_manifest"]).read_text(encoding="utf-8"))
actual = {p.name: sha256(p) for p in sorted(test_images.glob("*.jpeg"))}
assert actual == expected, "Official test images are missing, extra or different."
print(f"Verified {len(actual)} test images and the frozen release files.")


## Run detector and native mask inference

The detector emits scored masks at the original image size. The frozen rule combines or selects masks and serializes the result. The GPU stages run sequentially.

In [ ]:
detector_output = out / "detector-proposals.json"
detector_receipt = out / "detector-receipt.json"
detector_arguments = [str(root / value) if value.startswith("artifacts/") or value.startswith("scripts/") else value
                      for value in release["detector_arguments"]]
run(["-m", release["detector_module"], *detector_arguments, "--images", test_images,
     "--inventory", root / release["test_manifest"], "--config", root / release["detector_config"],
     "--split", root / release["detector_split"], "--training-receipt", root / release["detector_training_receipt"],
     "--training-source", root / release["detector_training_source"],
     "--checkpoint", root / release["detector_checkpoint"],
     "--settings-dir", out / "detector-settings", "--output", detector_output, "--receipt", detector_receipt])

csv_path = out / "submission.csv"
coverage_path = out / "coverage.json"
receipt_path = out / "inference.json"
run(["-m", "filament.detector_export", "--images", test_images,
     "--inventory", root / release["test_manifest"],
     "--detector-checkpoint", root / release["detector_checkpoint"], "--detector-receipt", detector_receipt,
     "--detections", detector_output, "--rule", root / release["detector_rule"],
     "--config", root / release["config"], "--checkpoint", root / release["checkpoint"],
     "--verifier", root / release["verifier"], "--output", csv_path,
     "--coverage", coverage_path, "--receipt", receipt_path, "--device", release["device"]])
assert sha256(coverage_path) == release["coverage_sha256"], "Coverage differs from the frozen export."


## Independently decode the export

The reviewer checks ID coverage, native dimensions, mask orientation and per-instance placement. A disk with zero predictions has no CSV rows; the coverage record still identifies the disk, and the release manifest verifies its input hash.


In [ ]:
qa_path = out / "qa.json"
run(["qa/verify_entry.py", "export", "--csv", csv_path, "--coverage", coverage_path,
     "--images", test_images, "--output", qa_path])
observed = sha256(csv_path)
print(f"CSV SHA-256: {observed}")
assert observed == release["submission_sha256"], "Export is structurally valid but differs from the frozen artifact."
print(f"Reproduced {csv_path} ({csv_path.stat().st_size:,} bytes).")


## Validation scope

The replay uses 180 unlabelled test JPEGs. The calibration and grouped evaluation results are recorded in the release and report. The Kaggle reference records official scoring.